# optimizer-loop-on-tensor — ex3: zero_grad per-param loop with set_to_none toggle

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-loop-on-tensor`. Running the final beacon cell reports progress against the `Optimizer: optimizer.step loop over params` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: optimizer.step loop over params` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-loop-on-tensor`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-loop-on-tensor"
DD_SUBTOPIC = "Optimizer: optimizer.step loop over params"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `zero_grad()` — the second per-parameter loop in every optimizer

Ex1 + ex2 implemented the per-parameter STEP loop. Every optimizer also has a per-parameter ZERO_GRAD loop. They share the same iteration pattern but do opposite things: step CONSUMES `.grad`, zero_grad CLEARS it for the next backward.

```python
@t.inference_mode()
def zero_grad(self, set_to_none: bool = True):
    for p in self.params:
        if p.grad is None:
            continue
        if set_to_none:
            p.grad = None              # cheaper: drops the tensor
        else:
            p.grad.zero_()             # in-place fill with zeros
```

**`set_to_none=True` is now the default in PyTorch.** Why: assigning `None` deallocates the `.grad` tensor's storage, saving memory between the `step()` and the next `backward()`. The next `backward()` re-allocates a fresh grad — which would have been overwritten anyway.

**`zero_()` (with underscore) is the in-place variant.** When the user wants `.grad` to remain a tensor of zeros (e.g. for downstream code that reads `.grad` between steps), pass `set_to_none=False`.

**The `if p.grad is None: continue` guard.** Same guard as in `step` — frozen / unused params have no `.grad` attribute. Without the guard, the `.grad.zero_()` call raises `AttributeError`.

### Exercise 3 — zero_grad per-param loop with set_to_none toggle

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the per-parameter `zero_grad` loop with a `set_to_none` toggle — `set_to_none=True` reassigns `p.grad = None` (memory win), `set_to_none=False` calls `p.grad.zero_()` (preserves the tensor).
> Keywords: zero-grad, set-to-none, per-param-loop, optimizer
> ```

**KCs targeted:** `zero-grad-per-param-loop`, `set-to-none-vs-zero_-tradeoff`

Implement `Ex3ZeroGradSGD`. A hand-rolled SGD that exposes both `step` and `zero_grad` as per-parameter loops.

Requirements:

1. `__init__(self, params, lr)`:
   - `self.params = list(params)`
   - `self.lr = lr`

2. `@t.inference_mode()` `step(self)`: standard SGD — for each `p in self.params` with `p.grad is not None`, do `p -= self.lr * p.grad`.

3. `@t.inference_mode()` `zero_grad(self, set_to_none: bool = True)`: for each `p in self.params`:
   - If `p.grad is None`, continue.
   - If `set_to_none` is True: `p.grad = None`.
   - Else: `p.grad.zero_()` (in-place zero, keeps the same tensor).

Constraints:
- `zero_grad(set_to_none=True)` must NOT use `.zero_()` — it must set the attribute to `None` directly.
- `zero_grad(set_to_none=False)` must preserve `p.grad`'s `data_ptr()` (the grad tensor stays at the same storage).
- Both branches must skip params with `p.grad is None`.

In [ ]:
class Ex3ZeroGradSGD:
    def __init__(self, params, lr):
        raise NotImplementedError()

    @t.inference_mode()
    def step(self):
        raise NotImplementedError()

    @t.inference_mode()
    def zero_grad(self, set_to_none: bool = True):
        raise NotImplementedError()


def _test_ex3():
    # === Construction ===
    p1 = t.tensor([1.0, 2.0, 3.0], requires_grad=True)
    p2 = t.tensor([4.0, 5.0], requires_grad=True)
    opt = Ex3ZeroGradSGD([p1, p2], lr=0.1)
    assert opt.params == [p1, p2] or (opt.params[0] is p1 and opt.params[1] is p2)
    assert opt.lr == 0.1

    # === Populate grads ===
    p1.grad = t.tensor([10.0, 20.0, 30.0])
    p2.grad = t.tensor([100.0, 200.0])
    ptr_g1 = p1.grad.data_ptr()
    ptr_g2 = p2.grad.data_ptr()

    # === step does SGD ===
    opt.step()
    assert t.allclose(p1, t.tensor([0.0, 0.0, 0.0]), atol=1e-6), (
        f'p1 after step: expected [0,0,0]; got {p1}'
    )
    assert t.allclose(p2, t.tensor([-6.0, -15.0]), atol=1e-6), (
        f'p2 after step: expected [-6,-15]; got {p2}'
    )

    # === zero_grad(set_to_none=False) preserves grad tensor identity ===
    opt.zero_grad(set_to_none=False)
    assert p1.grad is not None, 'set_to_none=False must KEEP grad tensor'
    assert p2.grad is not None, 'set_to_none=False must KEEP grad tensor'
    assert p1.grad.data_ptr() == ptr_g1, (
        'set_to_none=False must preserve grad storage (no realloc); '
        f'ptr was {ptr_g1}, now {p1.grad.data_ptr()}'
    )
    assert p2.grad.data_ptr() == ptr_g2, 'set_to_none=False must preserve p2 grad storage'
    assert t.all(p1.grad == 0), f'grad must be zeroed; got {p1.grad}'
    assert t.all(p2.grad == 0), f'grad must be zeroed; got {p2.grad}'

    # === zero_grad(set_to_none=True) drops the grad tensor entirely ===
    p1.grad = t.tensor([1.0, 2.0, 3.0])
    p2.grad = t.tensor([4.0, 5.0])
    opt.zero_grad(set_to_none=True)
    assert p1.grad is None, f'set_to_none=True must set grad to None; got {p1.grad}'
    assert p2.grad is None, f'set_to_none=True must set grad to None; got {p2.grad}'

    # === default is set_to_none=True (matches PyTorch's modern default) ===
    p1.grad = t.tensor([1.0, 2.0, 3.0])
    p2.grad = t.tensor([4.0, 5.0])
    opt.zero_grad()  # no arg
    assert p1.grad is None, 'default zero_grad must set grad to None'
    assert p2.grad is None

    # === None-grad guard: zero_grad on frozen param does NOT raise ===
    frozen = t.tensor([1.0, 1.0], requires_grad=False)
    active = t.tensor([1.0], requires_grad=True)
    active.grad = t.tensor([0.5])
    opt2 = Ex3ZeroGradSGD([frozen, active], lr=0.1)
    opt2.zero_grad(set_to_none=False)   # must not raise even though frozen.grad is None
    opt2.zero_grad(set_to_none=True)
    assert active.grad is None
    assert frozen.grad is None  # untouched

    # === step + zero_grad cycle drives the canonical training loop ===
    p = t.tensor([0.0], requires_grad=True)
    opt3 = Ex3ZeroGradSGD([p], lr=0.5)
    for _ in range(3):
        p.grad = t.tensor([2.0])  # fake backward
        opt3.step()
        opt3.zero_grad()
        assert p.grad is None  # cleared
    # p moved by -0.5 * 2.0 = -1.0 per step → final p = -3.0
    assert t.allclose(p, t.tensor([-3.0]), atol=1e-6), f'cycle drift: expected -3.0; got {p}'

    # === set_to_none=False inside the cycle ===
    p = t.tensor([0.0], requires_grad=True)
    opt4 = Ex3ZeroGradSGD([p], lr=0.5)
    p.grad = t.tensor([1.0])
    ptr_initial = p.grad.data_ptr()
    for _ in range(5):
        opt4.step()
        opt4.zero_grad(set_to_none=False)
        assert p.grad is not None
        assert p.grad.data_ptr() == ptr_initial, 'grad storage must persist with set_to_none=False'
        p.grad.add_(1.0)  # simulate next backward writing into the SAME grad tensor
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
class Ex3ZeroGradSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.inference_mode()
    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            p -= self.lr * p.grad

    @t.inference_mode()
    def zero_grad(self, set_to_none: bool = True):
        for p in self.params:
            if p.grad is None:
                continue
            if set_to_none:
                p.grad = None
            else:
                p.grad.zero_()
```

**`set_to_none=True` is now PyTorch's default.** Setting `p.grad = None` deallocates the grad tensor's storage. The next `backward()` re-allocates a fresh grad with the right shape — which the old grad's contents would have been overwritten by anyway. Net memory savings are real for large models.

**`.zero_()` (in-place) preserves storage.** The trailing underscore is PyTorch's in-place convention. The grad tensor's `data_ptr()` is unchanged, only its values are reset to 0. Use this when downstream code reads `.grad` between steps.

**The `if p.grad is None: continue` guard appears TWICE.** Once in `step` (don't apply update to frozen params), once in `zero_grad` (don't try to zero a non-existent attribute). Frozen / unused params hit both branches.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()